In [1]:
from pathlib import Path

# Laravel project root
PROJECT_ROOT = Path(r"C:\xampp\htdocs\iitm\booking")

# Controllers folder
CONTROLLERS_DIR = PROJECT_ROOT / "app" / "Http" / "Controllers"

print(f"Scanning: {CONTROLLERS_DIR}\n")

if not CONTROLLERS_DIR.exists():
    print("Controllers directory not found.")
    exit()

controllers = sorted(CONTROLLERS_DIR.rglob("*Controller.php"))

if not controllers:
    print("No controllers found.")
else:
    print(f"Found {len(controllers)} controller(s):\n")

    for i, controller in enumerate(controllers, start=1):
        relative = controller.relative_to(PROJECT_ROOT)
        print(f"{i:2}. {controller.stem}")
        print(f"    {relative}")

Scanning: C:\xampp\htdocs\iitm\booking\app\Http\Controllers

Found 4 controller(s):

 1. AdminController
    app\Http\Controllers\Admin\AdminController.php
 2. Controller
    app\Http\Controllers\Controller.php
 3. CustomerController
    app\Http\Controllers\Customer\CustomerController.php
 4. HomeController
    app\Http\Controllers\Home\HomeController.php


In [2]:
from flask import Flask, request, redirect, url_for, render_template_string, flash
from pathlib import Path
import threading

app = Flask(__name__)
app.secret_key = "secret"


# Laravel project path
PROJECT_ROOT = Path(r"C:\xampp\htdocs\iitm\booking")

CONTROLLERS_DIR = PROJECT_ROOT / "app" / "Http" / "Controllers"


INDEX_HTML = """
<h2>Laravel Controllers</h2>

<table border="1" cellpadding="8">

<tr>
<th>No</th>
<th>Controller</th>
<th>Path</th>
<th>Edit</th>
</tr>

{% for c in controllers %}

<tr>
<td>{{loop.index}}</td>
<td>{{c.name}}</td>
<td>{{c.relative}}</td>

<td>
<a href="/edit/{{c.relative}}">
Edit
</a>
</td>

</tr>

{% endfor %}

</table>
"""


EDITOR_HTML = """
<h3>{{filename}}</h3>

{% with messages=get_flashed_messages() %}
{% if messages %}
<p style="color:green">
{{messages[0]}}
</p>
{% endif %}
{% endwith %}


<form method="post">

<textarea 
name="content"
style="
width:100%;
height:700px;
font-family:Consolas;
font-size:15px;
">
{{content}}
</textarea>

<br><br>

<button>
Save
</button>

<a href="/">
Back
</a>

</form>
"""


def get_controllers():

    result=[]

    for file in CONTROLLERS_DIR.rglob("*Controller.php"):

        result.append({
            "name":file.stem,
            "relative":file.relative_to(PROJECT_ROOT).as_posix()
        })

    return sorted(result,key=lambda x:x["name"])



@app.route("/")
def index():

    return render_template_string(
        INDEX_HTML,
        controllers=get_controllers()
    )



@app.route("/edit/<path:filename>",methods=["GET","POST"])
def edit(filename):

    filepath = PROJECT_ROOT / filename


    if request.method=="POST":

        filepath.write_text(
            request.form["content"],
            encoding="utf-8"
        )

        flash("Saved")

        return redirect(
            url_for("edit",filename=filename)
        )


    content=filepath.read_text(
        encoding="utf-8"
    )


    return render_template_string(
        EDITOR_HTML,
        filename=filename,
        content=content
    )



def run_flask():

    app.run(
        host="127.0.0.1",
        port=5000,
        debug=False
    )


threading.Thread(
    target=run_flask
).start()

 * Tip: There are .env files present. Install python-dotenv to use them.


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [29/Jul/2026 22:11:55] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [29/Jul/2026 22:11:55] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [29/Jul/2026 22:11:58] "GET /edit/app/Http/Controllers/Admin/AdminController.php HTTP/1.1" 200 -
127.0.0.1 - - [29/Jul/2026 22:12:07] "POST /edit/app/Http/Controllers/Admin/AdminController.php HTTP/1.1" 302 -
127.0.0.1 - - [29/Jul/2026 22:12:07] "GET /edit/app/Http/Controllers/Admin/AdminController.php HTTP/1.1" 200 -
